In [ ]:
from DQNPlayer_HandBitmap import DQNPlayer_HandBitmap
from Deck import Deck
from Game import GameEnv as Game
from RandomPlayer import RandomPlayer
from DQNPlayer import DQNPlayer
from collections import OrderedDict
from bitarray import bitarray
from Player import Player
import itertools

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm

MAX_PLAYS = 6000

In [ ]:
################### UTILITIES ######################

from Deck import DeckHelper

def pad_msb_fixed(action: bitarray) -> bitarray:
    """
    Aggiunge esattamente 2 bit a 0 all'inizio (MSB) 
    per portare un bitarray di 14 bit a 16 bit (2 byte).
    """
    # Creiamo un bitarray di 2 zeri e lo concateniamo a sinistra
    return bitarray('00') + action

cumulated_rewards_learning_agent = []  # Initialize cumulative rewards for the learning agent

def print_game_state(game: Game, players_action: OrderedDict[Player, bitarray], played_cards: bitarray, rewards: dict[Player, int]):
    TOTAL_CARDS = game.deck.num_cards  # Assuming the deck has a num_cards attribute
    print(f"Current Game State:")
    for player in players_action.keys():
        print(f"Player {player.get_id()} - \t Cards Played: {show_bitarray_in_group(game, players_action[player], TOTAL_CARDS//4)}, Num Cards Played: {sum(players_action[player])}, Score: {rewards[player]}")
    print(f"Played Cards: \t      {played_cards}")
    print("-" * 50)

def show_bitarray_in_group(game: Game, bitarr: bitarray, group_size: int) -> str:
    """Helper function to display a bitarray in groups of a specified size for better readability."""
    groups = []
    
    # Iteriamo sui blocchi (es. di 13 in 13 per i semi)
    for start_idx in range(0, len(bitarr), group_size):
        group_bits = bitarr[start_idx : start_idx + group_size]
        cards_str = []
        
        # Iteriamo all'interno del singolo blocco
        for offset, bit in enumerate(group_bits):
            if bit == 1:
                # Calcoliamo l'indice assoluto della carta nel mazzo
                card_idx = start_idx + offset
                rank = DeckHelper.card_to_rank(game.deck, card_idx)
                cards_str.append(str(rank))
            else:
                cards_str.append('0')
                
        groups.append(' '.join(cards_str))
        
    return '    '.join(groups)


def repeat_and_average(n_runs: int = 5):
    """
    Decoratore che esegue la funzione di esperimento target 'n_runs' volte.
    - Fa la media elemento per elemento delle ricompense (gestendo lunghezze variabili).
    - Raggruppa le storie delle azioni in una lista (poiché i bitarray non sono mediabili).
    """
    def decorator(func):
        def wrapper(*args, **kwargs):
            all_rewards = []
            all_actions_runs: OrderedDict[int, list[list[bitarray]]] = OrderedDict() # Raccoglie la cronologia azioni intera di ogni singola run
            # OrderedDict[int, list[list[bitarray]]]]
            for run in range(n_runs):
                print(f"\n{'='*50}\n INIZIO ESECUZIONE SPERIMENTALE RUN {run + 1}/{n_runs}\n{'='*50}")
                
                rewards, actions = func(*args, **kwargs)
                all_rewards.append(rewards)
                all_actions_runs = actions  # Salva la cronologia delle azioni per questa run

            print(f"\n{'='*50}\n CALCOLO DELLA MEDIA DEI RISULTATI SU {n_runs} RUN...\n{'='*50}")

            averaged_rewards = OrderedDict()
            all_actions = OrderedDict()
            players = list(all_rewards[0].keys())
            player_round_action_list : list[list[bitarray]] = []

            for player in players:
                # --- GESTIONE REWARDS ---
                player_reward_lists = [run_rewards[player] for run_rewards in all_rewards]
                padded_rewards = list(itertools.zip_longest(*player_reward_lists, fillvalue=np.nan))
                
                # nanmean somma i reward per quello specifico round in tutte le run e lo divide per il numero di run
                mean_rewards = np.nanmean(padded_rewards, axis=1).tolist()
                averaged_rewards[player] = mean_rewards

            

            return averaged_rewards, all_actions_runs
        
        # Copia manuale dei metadati di base della funzione originale per evitare functools.wraps
        wrapper.__name__ = func.__name__
        wrapper.__doc__ = func.__doc__
        wrapper.__dict__.update(func.__dict__)
        return wrapper
    return decorator
###############################################


In [ ]:
################# GRAPHIC UTILITIES ###################
MOVING_WINDOW = 50
def plot_mean_stds(collections, collection_names, title, xlabel='Episodi', ylabel='Reward Totale Accumulato', ylimits: list[float] = [], ylog=False):
    """
    Sovrappone le medie di tutte le collezioni di dati in un unico grafico
    per facilitare il confronto diretto.
    
    Args:
        collezioni_dati: Lista di liste (es. [run1_data, run2_data, ...])
        titolo: Titolo del grafico unico.
        labels: Lista di etichette per la legenda (una per collezione).
    """
    n = len(collections)
        
    plt.figure(figsize=(10, 6))

    plt.rcParams.update({
        'font.size': 14,
        'axes.labelsize': 16,
        'axes.titlesize': 18,
        'legend.fontsize': 14,
        'xtick.labelsize': 14,
        'ytick.labelsize': 14,
        'lines.linewidth': 6,           # linea della media più spessa
        'grid.linewidth': 1.5,          # griglia più spessa
    })
    
    # Definiamo una palette di colori per distinguere le serie
    if n == 2:
        colors = ['#0072B2', '#D55E00']  # Scenario 1
    elif n == 4:
        colors = ["#d9eaff", "#66b3ff", "#0073e6", "#002b66"]
    elif n == 3:
        colors = [ "#66b3ff", "#0073e6", "#002b66"]  # Scenario 3
    else:
        # Fallback alla tua palette originale
        cmap = plt.get_cmap('tab10')
        colors = cmap(np.linspace(0, 1, n))
    
    for i, (data, label) in enumerate(zip(collections, collection_names)):
        smoothed_runs = np.array(data)
        media = np.mean(smoothed_runs, axis=0)
        std = np.std(smoothed_runs, axis=0)
        
        # Plot della media
        plt.plot(media, label=f'{label} (Media)', color=colors[i], linewidth=6)
        
        # Plot della deviazione standard (trasparente)
        plt.fill_between(
            range(len(media)),
            media - std,
            media + std,
            color=colors[i],
            alpha=0.15
        )
    
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.legend(loc='upper right')
    plt.ylim(ylimits if ylimits else None)
    if ylog:
        plt.yscale('log')
    plt.tight_layout()
    plt.show()
def moving_average(dati, window=10000):
    """Calcola la media mobile per smussare il rumore del grafico."""
    return [np.mean(dati[max(0, i - window + 1):i + 1]) for i in range(len(dati))]


In [ ]:
################### EXPERIMENT UTILITIES ###################

@repeat_and_average(n_runs=5)
def run_experiment(
    learning_agent: DQNPlayer, 
    TOTAL_PLAYERS: int, 
    MAX_PLAYS: int, 
    TOTAL_CARDS: int, 
    JACKS: int) -> tuple[OrderedDict[int, list[int]], OrderedDict[int, list[list[bitarray]]]]:
    """Esegue l'esperimento per un numero definito di episodi, tracciando le azioni, i risultati e le ricompense.
    Args:
        game (Game): L'ambiente di gioco.
        players (list[Player]): Lista dei giocatori coinvolti nell'esperimento, incluso l'agente di apprendimento.
        learning_agent (DQNPlayer): L'agente di apprendimento che si desidera monitorare.
        MAX_PLAYS (int): Il numero massimo di episodi da eseguire.
        TOTAL_CARDS (int): Il numero totale di carte nel gioco.
        JACKS (int): Il numero di jolly nel gioco.
    Returns:
        tuple[OrderedDict[Player, list[int]], OrderedDict[Player, list[list[bitarray]]]]: Una tupla contenente i cumuli di ricompense per ogni giocatore e la storia delle azioni per ogni round.
    """
    deck = Deck(num_cards=TOTAL_CARDS, jacks=JACKS)  # Exclude jokers for the deck
    game = Game(deck=deck)
    players: list[Player] = [RandomPlayer(player_id=i, total_cards=TOTAL_CARDS - JACKS) for i in range(TOTAL_PLAYERS - 1)]  # Create 3 random players
    players.append(learning_agent)  # Add the learning agent to the list of players
    rewards_for_player: OrderedDict[int, list[int]] = OrderedDict((int(player.get_id()), []) for player in players)  # Initialize cumulative rewards for each player
    for player in players:
        game.add_player(player)  # Ensure all players are added to the game environment

    actions_history: OrderedDict[int, list[list[bitarray]]] = OrderedDict((int(player.get_id()), []) for player in players)  
    # For each player, for each round in an episode, we will store the action taken as a bitarray in a list. 
    for episode in range(MAX_PLAYS):
        print(f"\n\nEpisode {episode + 1}/{MAX_PLAYS}")
        game.reset()  # Reset the game environment for a new episode
        game.start_game()  # Start the game

        round = 0
        episode = 0
        total_cards_played = bitarray(TOTAL_CARDS)  # Initialize a bitarray to keep track of all played cards in the current episode
        
        while True:
            print(f"Round {episode + 1}")
            [print(f"Player {player.get_id()} hand cards left:\t{sum(player._hand)}") for player in game.players]  # Debug: Print each player's hand before playing

            actions = run_actions(game, players, learning_agent)  # Get actions for all players
            for player in players:
                if len(actions_history[int(player.get_id())]) <= round:
                    actions_history[int(player.get_id())].append([])  # Initialize the list for this round if it doesn't exist
                actions_history[int(player.get_id())][round].append(actions[player])  # Store the action taken by each player in the history

            game_results, _, done, truncated, _ = game.step(actions)
            
            print(f"Played cards: {game_results['played_cards']}")  # Debug: Print the results of the game step
            print(f"total_cards_played before update: {total_cards_played}")  # Debug: Print the total cards played before updating
            total_cards_played |= game_results["played_cards"]  # Update the total cards played
            print_game_state(game, actions, total_cards_played, game_results["rewards"])

            update_players_state(players, actions, game_results)  # Update the state of each player based on the actions and results


            [rewards_for_player[int(player.get_id())].append(game_results["rewards"][player]) for player in players]  # Track cumulative rewards for each player

            if done or truncated:
                print(f"Game Over\n")
                round = 0
                break
            round += 1
            episode += 1
    return rewards_for_player, actions_history  # Return the cumulative rewards for the learning agent after all episodes


def run_actions(game: Game, players: list[Player], learning_agent: DQNPlayer) -> OrderedDict[Player, bitarray]:
    """
    Esegue le azioni dei giocatori e restituisce un dizionario ordinato con le azioni.
    """

    actions = OrderedDict()
    played_cards = 0  # Contatore delle carte giocate in questo round prima del turno corrente

    for player in players:
        # CORREZIONE 1: Confronto diretto per identità di memoria (is)
        # Sostituisce il confronto lento tramite id: 'player.get_id() == learning_agent.get_id()'
        if player is learning_agent:
            learning_agent.num_cards_played = played_cards
            
        # Otteniamo la giocata del giocatore (che leggerà num_cards_played aggiornato)
        action = player.play_cards()
        actions[player] = action
        
        # CORREZIONE 2: Sostituito il lentissimo sum(action) con il C-optimized action.count()
        # 'sum(bitarray)' costringe Python a spacchettare ogni bit in un oggetto Bool/Int e sommarli a livello software.
        # '.count()' della libreria bitarray esegue il calcolo direttamente in C a livello hardware.
        played_cards += action.count()

    return actions


def update_players_state(players: list[Player], actions: OrderedDict[Player, bitarray], game_results: dict):
    """
    Aggiorna lo stato di ogni giocatore in base alle azioni e ai risultati del gioco.
    """
    for player in players:
        player.update_state(
            actions[player],
            game_results["played_cards"], 
            game_results["rewards"][player]
        )






In [ ]:
############################# 4 x 4 EXPERIMENT SETTING #############################
TOTAL_PLAYERS = 4
CARDS_PER_PLAYER = 4
JACKS = 0
TOTAL_CARDS = TOTAL_PLAYERS * CARDS_PER_PLAYER  # Total cards including jokers

# ---------------- 4 x 4 AGENT --------------------- #
AGENT_4x4 = DQNPlayer_HandBitmap(
    player_id=TOTAL_PLAYERS - 1,  # Assign the last player ID to the learning agent
    total_cards=TOTAL_CARDS, 
    cards_per_player=TOTAL_CARDS // TOTAL_PLAYERS,  # Calculate cards per player based on total cards and players
    discount_factor=0.99,
    batch_size=64,
    SYNC_TARGET_EVERY=400,
    replay_buffer_capacity=MAX_PLAYS
)

rewards_for_player_4x4, actions_history_4x4 = run_experiment(
    AGENT_4x4,
    TOTAL_PLAYERS, MAX_PLAYS, TOTAL_CARDS, JACKS=JACKS
)
# ----------------------------------------------------------- #

In [ ]:
############################# 6 x 4 EXPERIMENT SETTING #############################
TOTAL_PLAYERS = 6
CARDS_PER_PLAYER = 4
TOTAL_CARDS = TOTAL_PLAYERS * CARDS_PER_PLAYER  # Total cards including jokers
JACKS = 0

# ---------------- 6 x 4 AGENT --------------------- #
AGENT_6x4 = DQNPlayer_HandBitmap(
    player_id=TOTAL_PLAYERS - 1,  # Assign the last player ID to the learning agent
    total_cards=TOTAL_CARDS, 
    cards_per_player=TOTAL_CARDS // TOTAL_PLAYERS,  # Calculate cards per player based on total cards and players
    discount_factor=0.99,
    batch_size=64,
    SYNC_TARGET_EVERY=400,
    replay_buffer_capacity=MAX_PLAYS
)

rewards_for_player_6x4, actions_history_6x4 = run_experiment(
    AGENT_6x4,
    TOTAL_PLAYERS, MAX_PLAYS, TOTAL_CARDS, JACKS=JACKS
)

# ----------------------------------------------------------- #

In [ ]:
############################# 8 x 4 EXPERIMENT SETTING #############################
TOTAL_PLAYERS = 8
CARDS_PER_PLAYER = 4
TOTAL_CARDS = TOTAL_PLAYERS * CARDS_PER_PLAYER  # Total cards including jokers
JACKS = 0

# ---------------- 8 x 4 AGENT --------------------- #
AGENT_8x4 = DQNPlayer_HandBitmap(
    player_id=TOTAL_PLAYERS - 1,  # Assign the last player ID to the learning agent
    total_cards=TOTAL_CARDS, 
    cards_per_player=TOTAL_CARDS // TOTAL_PLAYERS,  # Calculate cards per player based on total cards and players
    discount_factor=0.99,
    batch_size=64,
    SYNC_TARGET_EVERY=400,
    replay_buffer_capacity=MAX_PLAYS
)

rewards_for_player_8x4, actions_history_8x4 = run_experiment(
    AGENT_8x4,
    TOTAL_PLAYERS, MAX_PLAYS, TOTAL_CARDS, JACKS=JACKS
)

# ----------------------------------------------------------- #

In [ ]:
############################# 12 x 4 EXPERIMENT SETTING #############################
TOTAL_PLAYERS = 12
CARDS_PER_PLAYER = 4
TOTAL_CARDS = TOTAL_PLAYERS * CARDS_PER_PLAYER  # Total cards including jokers
JACKS = 0

# ---------------- 12 x 4 AGENT --------------------- #
AGENT_12x4 = DQNPlayer_HandBitmap(
    player_id=TOTAL_PLAYERS - 1,  # Assign the last player ID to the learning agent
    total_cards=TOTAL_CARDS, 
    cards_per_player=TOTAL_CARDS // TOTAL_PLAYERS,  # Calculate cards per player based on total cards and players
    discount_factor=0.99,
    batch_size=64,
    SYNC_TARGET_EVERY=400,
    replay_buffer_capacity=MAX_PLAYS
)

rewards_for_player_12x4, actions_history_12x4 = run_experiment(
    AGENT_12x4,
    TOTAL_PLAYERS, MAX_PLAYS, TOTAL_CARDS, JACKS=JACKS
)

# ----------------------------------------------------------- #

In [ ]:
############# PLOT ESTIMATION ERRORS #############
MOVING_WINDOW = 500
estimation_errors_4x4, target_errors_4x4 = AGENT_4x4.get_errors()  # Get the estimation and target errors from the learning agent
estimation_errors_6x4, target_errors_6x4 = AGENT_6x4.get_errors()  # Get the estimation and target errors from the learning agent
estimation_errors_8x4, target_errors_8x4 = AGENT_8x4.get_errors()  # Get the estimation and target errors from the learning agent
estimation_errors_12x4, target_errors_12x4 = AGENT_12x4.get_errors()  # Get the estimation and target errors from the learning agent

plot_mean_stds(
    [
        [moving_average(target_errors_4x4, window=MOVING_WINDOW)],
        [moving_average(target_errors_6x4, window=MOVING_WINDOW)], 
        [moving_average(target_errors_8x4, window=MOVING_WINDOW)], 
        [moving_average(target_errors_12x4, window=MOVING_WINDOW)]
    ], 
    collection_names=["4x4 Agent", "6x4 Agent", "8x4 Agent", "12x4 Agent"],
    title="Target Values Comparison",
    xlabel="Episodes",
    ylabel="Average Target Value",
)  # Passiamo una lista di run, anche se ne abbiamo solo una




In [ ]:
MOVING_WINDOW = 200
TD_difference_4x4 = [estimation_errors_4x4[i] - target_errors_4x4[i] for i in range(len(target_errors_4x4))]  # Create a list of indices for the x-axis
TD_difference_6x4 = [estimation_errors_6x4[i] - target_errors_6x4[i] for i in range(len(target_errors_6x4))]  # Create a list of indices for the x-axis
TD_difference_8x4 = [estimation_errors_8x4[i] - target_errors_8x4[i] for i in range(len(target_errors_8x4))]  # Create a list of indices for the x-axis
TD_difference_12x4 = [estimation_errors_12x4[i] - target_errors_12x4[i] for i in range(len(target_errors_12x4))]  # Create a list of indices for the x-axis
plot_mean_stds(
    [
        [moving_average(TD_difference_4x4, window=MOVING_WINDOW)],
        [moving_average(TD_difference_6x4, window=MOVING_WINDOW)],
        [moving_average(TD_difference_8x4, window=MOVING_WINDOW)],
        [moving_average(TD_difference_12x4, window=MOVING_WINDOW)], 
    ], 
    collection_names=["TD Error (4x4)", "TD Error (6x4)", "TD Error (8x4)", "TD Error (12x4)"],
    title="TD Errors Comparison (3x8 vs 3x12 vs 3x16)",
    xlabel="Episodes",
    ylabel="Average Error per Episode",
)  # Passiamo una lista di run, anche se ne abbiamo solo una

In [ ]:
################ PLOT REWARDS ################
MOVING_WINDOW = 500  # Ad esempio, se MAX_PLAYS è 100, una finestra di 10 episodi per la media mobile
plot_mean_stds(
    [
        [moving_average(rewards_for_player_4x4[AGENT_4x4.get_id()], window=MOVING_WINDOW)], 
        [moving_average(rewards_for_player_4x4[0], window=MOVING_WINDOW)], 
    ], 
    collection_names=["Learning Player Rewards","Random Player Rewards"],
    title="Learning vs Random: Rewards Comparison (4x4)",
    xlabel="Episodes",
    ylabel="Average Rewards per Episode",
)  # Passiamo una lista di run, anche se ne abbiamo solo una
plot_mean_stds(
    [
        [moving_average(rewards_for_player_6x4[AGENT_6x4.get_id()], window=MOVING_WINDOW)], 
        [moving_average(rewards_for_player_6x4[0], window=MOVING_WINDOW)],
    ], 
    collection_names=["Learning Player Rewards","Random Player Rewards"],
    title="Learning vs Random: Rewards Comparison (6x4)",
    xlabel="Episodes",
    ylabel="Average Rewards per Episode",
)  # Passiamo una lista di run, anche se ne abbiamo solo una

plot_mean_stds(
    [
        [moving_average(rewards_for_player_8x4[AGENT_8x4.get_id()], window=MOVING_WINDOW)], 
        [moving_average(rewards_for_player_8x4[0], window=MOVING_WINDOW)],
    ], 
    collection_names=["Learning Player Rewards","Random Player Rewards"],
    title="Learning vs Random: Rewards Comparison (8x4)",
    xlabel="Episodes",
    ylabel="Average Rewards per Episode",
)  # Passiamo una lista di run, anche se ne abbiamo solo una
plot_mean_stds(
    [
        [moving_average(rewards_for_player_12x4[AGENT_12x4.get_id()], window=MOVING_WINDOW)], 
        [moving_average(rewards_for_player_12x4[0], window=MOVING_WINDOW)], 
    ], 
    collection_names=["Learning Player Rewards","Random Player Rewards"],
    title="Learning vs Random: Rewards Comparison (12x4)",
    xlabel="Episodes",
    ylabel="Average Rewards per Episode",
)  # Passiamo una lista di run, anche se ne abbiamo solo una

